In [49]:
import flwr as fl
print(fl.__version__)

1.30.0


In [51]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

X_train_t.shape

torch.Size([168834, 53])

In [53]:
class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [55]:
NUM_CLIENTS = 5

def partition_data(X, y, num_clients):
    idx = np.random.permutation(len(X))  # shuffling first
    splits = np.array_split(idx, num_clients)  # splitting into N chunks
    return [(X[s], y[s]) for s in splits]  # pairing X and y per client

client_data = partition_data(X_train_t.numpy(), y_train_t.numpy(), NUM_CLIENTS)
[d[0].shape for d in client_data]

[(33767, 53), (33767, 53), (33767, 53), (33767, 53), (33766, 53)]

In [57]:
from flwr.client import NumPyClient, ClientApp
from flwr.common import Context
from collections import OrderedDict

class FlowerClient(NumPyClient):
    def __init__(self, model, X, y):
        self.model = model
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def _get_params(self):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def _set_params(self, params):
        params_dict = zip(self.model.state_dict().keys(), params)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def get_parameters(self, config):
        return self._get_params()

    def fit(self, parameters, config):
        self._set_params(parameters)  # loading server's weights first
        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)

        self.model.train()
        for _ in range(3):  # 3 local epochs per round now
            optimizer.zero_grad()
            outputs = self.model(self.X)
            loss = criterion(outputs, self.y)
            loss.backward()
            optimizer.step()  # updating locally

        return self._get_params(), len(self.X), {}

    def evaluate(self, parameters, config):
        self._set_params(parameters)  # using latest global weights
        criterion = nn.BCELoss()
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(self.X)
            loss = criterion(outputs, self.y).item()
            preds = (outputs > 0.5).float()
            acc = (preds == self.y).float().mean().item()  # checking accuracy
        return loss, len(self.X), {"accuracy": acc}

In [59]:
def client_fn(context: Context):
    partition_id = int(context.node_config["partition-id"])  # which client am I
    X, y = client_data[partition_id]  # grabbing my slice of data

    model = IDSNet(input_dim=X.shape[1])  # fresh model per client
    return FlowerClient(model, X, y).to_client()

client_app = ClientApp(client_fn=client_fn)

In [61]:
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg

def weighted_average(metrics):
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}

def server_fn(context: Context):
    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_available_clients=NUM_CLIENTS,
        evaluate_metrics_aggregation_fn=weighted_average,
    )
    config = ServerConfig(num_rounds=15)
    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

In [63]:
from flwr.simulation import run_simulation

run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
    backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 0}},
)


            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower ServerApp, config: num_rounds=15, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
INFO :      Received initial parameters from one random client
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
(pid=gcs_server) [2026-09-21 18:43:13,932 E 23708 7936] (gcs_server.exe) gcs_server.cc:302: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 5 clients (ou

In [65]:
# saving these results so I can reference them later
fed_iid_results = {
    "final_accuracy": 0.9209,
    "rounds": 15,
    "accuracy_history": [0.2368, 0.2376, 0.2380, 0.2415, 0.3989, 0.5178, 0.7452,
                          0.8682, 0.8927, 0.9291, 0.9369, 0.9207, 0.9203, 0.9204, 0.9209]
}
import json
with open('federated_iid_results.json', 'w') as f:
    json.dump(fed_iid_results, f)